<a href="https://colab.research.google.com/github/ranjanvrma/retail-sales-analytics/blob/main/notebooks/04_advanced_business_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import numpy as np
import pandas as pd
import seaborn as sns

In [29]:
df = pd.read_csv('/content/superstore_sales.csv')

In [30]:
df.isnull().sum()

,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


In [31]:
df.dropna(inplace = True)

In [32]:
df.isnull().sum()

,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


In [33]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


## Customer Insights

In [34]:
# Top 20% customers contribute what % of revenue? (Pareto analysis)
customer_sales = df.groupby(['Customer ID', 'Customer Name'])['Sales'].sum().sort_values(ascending = False)

top_20_count = int(np.ceil(len(customer_sales) * 0.20))

top_20_sales = customer_sales.head(top_20_count).sum()

total_revenue = customer_sales.sum()

pareto_percentage = (top_20_sales / total_revenue) * 100

print(f"Top 20% customers contribute {pareto_percentage:.2f}% of revenue.")

Top 20% customers contribute 48.59% of revenue.


In [35]:
# Customers who purchased only once.
total_count = df['Customer ID'].value_counts().reset_index()

one_time = total_count[total_count['count'] == 1]

one_time

,Customer ID,count
787,JR-15700,1
788,AO-10810,1
789,LD-16855,1
790,CJ-11875,1
791,SC-20845,1
792,RE-19405,1


In [36]:
# Customers who purchased more than 10 times.
total_count = df['Customer ID'].value_counts().reset_index()

more_than_10 = total_count[total_count['count'] > 10]

more_than_10

,Customer ID,count
0,WB-21850,35
1,PP-18955,34
2,MA-17560,34
3,JL-15835,33
4,CK-12205,32
...,...,...
437,MV-18190,11
438,JL-15850,11
439,TB-21175,11
440,TB-21280,11


In [37]:
# Average orders per customer.
np.floor(df.groupby('Customer ID')['Order ID'].count().mean())

np.float64(12.0)

## Product Insights

In [38]:
# Which products have high sales but low order frequency?
product_stats = df.groupby('Product Name').agg(
    Total_Sales = ('Sales', 'sum'),
    Order_Frequency = ('Order ID', 'nunique')
).reset_index()

avg_sales = product_stats['Total_Sales'].mean()
avg_orders = product_stats['Order_Frequency'].mean()

final_products = product_stats[
    (product_stats['Total_Sales'] > avg_sales) &
    (product_stats['Order_Frequency'] < avg_orders)
]
final_products

,Product Name,Total_Sales,Order_Frequency
14,"24 Capacity Maxi Data Binder Racks, Pearl",3537.240,4
17,3.6 Cubic Foot Counter Height Office Refrigerator,2946.200,5
19,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.890,2
20,"3D Systems Cube Printer, 2nd Generation, White",2339.982,2
25,"3M Polarizing Task Lamp with Clamp Arm, Light ...",2191.680,4
...,...,...,...
1727,Xerox 1941,1740.510,5
1832,Xerox WorkCentre 6505DN Laser Multifunction Pr...,2519.958,1
1833,Xiaomi Mi3,2279.960,1
1837,Zebra GX420t Direct Thermal/Thermal Transfer P...,5787.355,3


In [39]:
# Which products sell frequently but generate low revenue?
product_stats = df.groupby('Product Name').agg(
    Total_Sales = ('Sales', 'sum'),
    Order_Frequency = ('Order ID', 'nunique')
).reset_index()

avg_sales = product_stats['Total_Sales'].mean()
avg_orders = product_stats['Order_Frequency'].mean()

final_products = product_stats[
    (product_stats['Order_Frequency'] > avg_orders) &
    (product_stats['Total_Sales'] < avg_sales)
]

final_products

,Product Name,Total_Sales,Order_Frequency
3,"#10 White Business Envelopes,4 1/8 x 9 1/2",379.214,6
4,"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",286.672,10
5,"#10- 4 1/8"" x 9 1/2"" Security-Tint Envelopes",146.688,8
11,12-1/2 Diameter Round Wall Clock,551.448,8
15,24-Hour Round Wall Clock,487.512,6
...,...,...,...
1834,"XtraLife ClearVue Slant-D Ring Binder, White, 3""",386.084,10
1839,Zebra Zazzle Fluorescent Highlighters,100.928,6
1840,Zipper Ring Binder Pockets,81.744,13
1846,invisibleSHIELD by ZAGG Smudge-Free Screen Pro...,442.554,7


In [40]:
# Category share within each region.
df.groupby(['Region', 'Category'])['Row ID'].count() / df.groupby('Region')['Row ID'].count() * 100

Region   Category       
Central  Furniture          20.641195
         Office Supplies    61.440492
         Technology         17.918314
East     Furniture          21.232877
         Office Supplies    59.877433
         Technology         18.889690
South    Furniture          20.400501
         Office Supplies    61.514393
         Technology         18.085106
West     Furniture          22.006369
         Office Supplies    59.235669
         Technology         18.757962
Name: Row ID, dtype: float64

In [41]:
# Top product in every state.
df.groupby([df['State'] == 'Arizona', 'Product Name']).size()

State  Product Name                                                       
False  "While you Were Out" Message Book, One Form per Page                    3
       #10 Gummed Flap White Envelopes, 100/Box                                4
       #10 Self-Seal White Envelopes                                           4
       #10 White Business Envelopes,4 1/8 x 9 1/2                              6
       #10- 4 1/8" x 9 1/2" Recycled Envelopes                                10
                                                                              ..
True   Xerox 210                                                               1
       Xerox 230                                                               1
       Xerox 4200 Series MultiUse Premium Copy Paper (20Lb. and 84 Bright)     1
       iHome FM Clock Radio with Lightning Dock                                1
       netTALK DUO VoIP Telephone Service                                      1
Length: 2047, dtype: int64

## Geographic Insights

In [42]:
# Which cities contribute 80% of total sales?
city_sales = df.groupby('City')['Sales'].sum().sort_values(ascending = False)

top_80_count = int(np.ceil(len(city_sales) * 0.80))

top_80_sales = city_sales.head(top_80_count)

top_80_sales

,Sales
City,
New York City,252462.547
Los Angeles,173420.181
Seattle,116106.322
San Francisco,109041.120
Philadelphia,108841.749
...,...
Laurel,152.590
Champaign,151.960
Thomasville,151.292


In [43]:
# Which state has the highest average order value?
df.groupby('State')['Sales'].mean().sort_values(ascending = False).head(10)

,Sales
State,
Wyoming,1603.136000
Nevada,428.951333
Rhode Island,409.545927
Montana,372.623467
Indiana,360.877037
Missouri,336.441667
Minnesota,335.541011
Alabama,319.846557
Virginia,315.342500


In [49]:
# Region-wise customer count.
df.groupby('Region')['Customer ID'].nunique()

,Customer ID
Region,
Central,626
East,669
South,509
West,681


## Time Insights

In [51]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format = '%d/%m/%Y')

In [70]:
# Sales growth from year to year (%).
yearly_sales = df.groupby(df['Order Date'].dt.year)['Sales'].sum().reset_index(name = 'Sales')

yearly_sales['Sales Growth(%)'] = (yearly_sales['Sales'].pct_change() * 100).round(2)

yearly_sales

,Order Date,Sales,Sales Growth(%)
0,2015,479856.2081,NaN
1,2016,454315.9054,-5.32
2,2017,597225.4900,31.46
3,2018,721209.8092,20.76


In [80]:
# Month-over-month growth.
monthly_sales = df.groupby(df['Order Date'].dt.month)['Sales'].sum().reset_index(name = 'Sales')

monthly_sales['Order Date'] = pd.to_datetime(monthly_sales['Order Date'], format = '%m').dt.month_name()

monthly_sales['Sales Growth(%)'] = (monthly_sales['Sales'].pct_change() * 100).round(2)

monthly_sales

,Order Date,Sales,Sales Growth(%)
0,January,91982.1396,NaN
1,February,59371.1154,-35.45
2,March,197573.5872,232.78
3,April,134988.2506,-31.68
4,May,154086.7237,14.15
5,June,145837.5233,-5.35
6,July,145535.6890,-0.21
7,August,157315.9270,8.09
8,September,300103.4117,90.76
9,October,199496.2947,-33.52


In [81]:
# Quarter contribution (%).
quarter_sales = df.groupby(df['Order Date'].dt.quarter)['Sales'].sum().reset_index(name = 'Sales')

quarter_sales['Sales Growth(%)'] = (quarter_sales['Sales'].pct_change() * 100).round(2)

quarter_sales

,Order Date,Sales,Sales Growth(%)
0,1,348926.8422,NaN
1,2,434912.4976,24.64
2,3,602955.0277,38.64
3,4,865813.0452,43.59
